In [68]:
import sys
import os
import networkx as nx
from typing import Hashable, TypeAlias
from tqdm import tqdm
import gc


In [69]:
CLASSES_PATH = os.path.dirname(os.path.abspath('D:/Code/Classes/'))
if not (CLASSES_PATH in sys.path):
    sys.path.append(CLASSES_PATH)
from Classes.Files_Handler_Class import Files_Handler
from Classes.Bcolors_Class import Bcolors as bcolors
from Classes.Random_Walk import Random_Walk
from Classes.Generate_Embedings import Generate_Embedings
from Classes.Load_Graph import Load_Graph

In [70]:
Random_Walk = Random_Walk()
Gnerate_Embedings = Generate_Embedings()
Load_Graph = Load_Graph()

In [71]:
Node: TypeAlias = Hashable  # Alias for readability

In [72]:
files_handler_obj = Files_Handler()
networks_list = []
networks_root_dir = ''
if networks_root_dir == '':
    networks_root_dir = files_handler_obj.select_dir() + '/'
# networks_root_dir = 'D:/Masters thesis/Networks Dataset/Monoplex/Test/'
networks_files = files_handler_obj.get_files_by_extensions(networks_root_dir, ['.edgeslist', '.edges', '.txt'])


In [73]:
networks_files

['D:/Datasets/IM/Multilayer/00 - General Forms/UCINET IV Datasets/Kapferer tailor shop - Copy/Kapferer tailor shop.edgeslist']

In [74]:
task_type = "CD"
rw_type = 'Vector'
rw_method = "Frequent Nodes"
embedding_attribute = 'Label'
compression_method = 'lzma'
if embedding_attribute == 'Label' and rw_method == 'Scale':
    sys.exit(0)
if rw_method == "Frequent Nodes":
    rw_length = 512

In [75]:
final_rw_length = 512
embeddings_dimension = 128
walk_depth = 3
if rw_type == 'Vector':
    num_walks = 1
else:
    num_walks = 64
window = 3
min_count = 1
workers = 4
epochs = 100
batch_words = 4
p, q = 1, 0.5,

In [76]:
network_dataset_type = "Alaska Master" # "Alaska Master" or "LFR" or "Multiplex Edges"
delimiter = " " # " " or ","
if network_dataset_type == "LFR":
    networks_files_copy = networks_files.copy()
    for item in networks_files_copy:
        if item.split("_")[-1] == "Community.txt":
            networks_files.remove(item)


In [77]:
networks_files

['D:/Datasets/IM/Multilayer/00 - General Forms/UCINET IV Datasets/Kapferer tailor shop - Copy/Kapferer tailor shop.edgeslist']

In [78]:
networks_info = {}
networks_random_walks = {}
accepable_files = [".edgeslist", ".edges", ".txt", '.edgelist']
print(bcolors.WARNING + f"Networks load as a list of Geaphs.\n" + bcolors.ENDC)
graphs_of_networks_with_random_walk = {}
graphs_of_networks_without_random_walk = {}
i = 1
for item in networks_files:
    file_info = files_handler_obj.get_file_path_info(item)
    networks_info[file_info['name']] = file_info
    if file_info["type"] in accepable_files:
        print(f"{i}- {file_info['name']}")
        graphs_of_network = None
        if file_info['type'] == ".mat":
            graphs_of_network, nodes_list, labels = Load_Graph.load_multilayer_graph(file_path=item, network_type="Matlab File", 
                                                                directed=False, delimiter=None, tabs="", print_layer_info=False)
                
        elif file_info['type'] == ".edges" or file_info['type'] == ".txt" or file_info['type'] == ".edgeslist":
            graphs_of_network, nodes_list, labels = Load_Graph.load_multilayer_graph(file_path=item, network_type="Alaska Master", 
                                                                directed=False, delimiter=None, tabs="", print_layer_info=False)
        print(len(graphs_of_network))
        random_walk_load_status, networks_random_walks[file_info['name']] = Random_Walk.load_multilayer_network_nodes_random_walk(file_info, rw_method, rw_type,
                                                              final_rw_length, walk_depth, num_walks,
                                                              compression_method, tabs='\t')
        if random_walk_load_status != False:
            graphs_of_networks_with_random_walk[file_info['name']] = graphs_of_network
        else :
            graphs_of_networks_without_random_walk[file_info['name']] = graphs_of_network

        i += 1

Networks load as a list of Geaphs.

1- Kapferer tailor shop
Network with 39 nodes and 4 layers loaded successfully.
4
	Random walk file not found.


In [79]:
graphs_of_networks_without_random_walk_count = len(graphs_of_networks_without_random_walk)
print(f"{bcolors.green_fg}{bcolors.bold}Graphs with random walk: {bcolors.end_color}{bcolors.bold}{len(graphs_of_networks_with_random_walk)}{bcolors.end_color}")
print(f"{bcolors.red_fg}{bcolors.bold}Graphs without random walk: {bcolors.end_color}{bcolors.bold}{len(graphs_of_networks_without_random_walk)}{bcolors.end_color}")


Graphs with random walk: 0
Graphs without random walk: 1


In [80]:
def sort_list_by_frequency(input_list:list):
    frequency_dict = {}
    for item in input_list:
        if item in frequency_dict:
            frequency_dict[item] += 1
        else:
            frequency_dict[item] = 1
    sorted_items = sorted(frequency_dict.items(), key=lambda x: x[1], reverse=True)
    sorted_list = [item[0] for item in sorted_items]
    return sorted_list

In [81]:
gc.collect()
i = 1
for networks_name, graphs_of_network in graphs_of_networks_without_random_walk.items():
    print(f"{i}- {networks_name}")
    if rw_type == 'Vector':
        networks_random_walks[networks_info[networks_name]['name']] = Random_Walk.multilayer_graph_random_walk_vector(graphs_of_network, rw_length, walk_depth, attribute='label')
    print(f"\tgraph random walk size: {len(networks_random_walks[networks_info[networks_name]['name']])}")

        # ----------------- Frequent Nodes --------------------------------
    pbar = tqdm(total=len(networks_random_walks[networks_info[networks_name]['name']]))
    pbar.set_description(f"\tExtract Frequent Nodes")
    pbar.unit = ' Node'
    pbar.colour = 'Blue'
    for node, node_random_walks in networks_random_walks[networks_info[networks_name]['name']].items():
        for layer, random_walk in node_random_walks.items():
            maim_randome_walk = [random_walk[0]] # For main node stay first element
            maim_randome_walk.extend(sort_list_by_frequency(random_walk))
            if len(maim_randome_walk) > final_rw_length:
                maim_randome_walk = maim_randome_walk[:final_rw_length]
            elif  len(maim_randome_walk) < final_rw_length:
                maim_randome_walk.extend(random_walk[1:(final_rw_length - len(maim_randome_walk)+1)])
        pbar.update(1)
    pbar.close()
    #------------------------------------------------------------------        
    
    Random_Walk.write_multilayer_network_nodes_random_walk(networks_random_walks[networks_info[networks_name]['name']], networks_info[networks_name],
                                rw_method, rw_type,
                                  final_rw_length, walk_depth, num_walks,
                                  compression_method, tabs='\t')

    i += 1

1- Kapferer tailor shop
	Create nodes random walk vector:
		walk_length: 512, walk_depth: 3


		Layer 3   : 100%|██████████| 4/4 [00:00<00:00, 28.76 Layer/s]


	graph random walk size: 39


	Extract Frequent Nodes: 100%|██████████| 39/39 [00:00<00:00, 2600.35 Node/s]

	Write Kapferer tailor shop nodes random walk in file using lzma compression method.


		Write don.
		File path: D:/Datasets/IM/Multilayer/00 - General Forms/UCINET IV Datasets/Kapferer tailor shop - Copy/Kapferer tailor shop/
		File name: Kapferer tailor shop nodes Frequent Nodes random walk Vector walk_length=512 walk_depth=3 num_walks=1.xz


In [82]:
graphs_of_networks_without_random_walk_count = len(graphs_of_networks_without_random_walk)
for networks_name, _ in graphs_of_networks_without_random_walk.items():
    graphs_of_networks_with_random_walk[networks_name] = graphs_of_networks_without_random_walk[networks_name]
graphs_of_networks_without_random_walk = {}
print(f"{bcolors.green_fg}{bcolors.bold}Graphs with random walk: {bcolors.end_color}{bcolors.bold}{len(graphs_of_networks_with_random_walk)}{bcolors.end_color}")
print(f"{bcolors.red_fg}{bcolors.bold}Graphs without random walk: {bcolors.end_color}{bcolors.bold}{len(graphs_of_networks_without_random_walk)}{bcolors.end_color}")

networks_graphs = graphs_of_networks_with_random_walk

Graphs with random walk: 1
Graphs without random walk: 0


In [83]:
import winsound
winsound.Beep(500, 750)
# os.system('shutdown -s')